# Agentic AI Domain Q&A: Fine-tuning vs Adapters vs RAG
This notebook compares three domain specialization strategies:
- Fine-tuning (simulated with an augmented system prompt)
- Adapter-like specialization (prompt-based adapter simulation)
- RAG (retrieval + generation)

**Knowledge Base (KB)**
- Agentic AI agents use memory, tools, and goals to act.
- LangChain and CrewAI are popular frameworks for building AI agents.
- Retrieval-Augmented Generation (RAG) improves accuracy by fetching external knowledge.

**Questions**
1. What are the key components of Agentic AI?
2. Name one framework for AI agents.
3. How does RAG improve answers?

In [ ]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

kb = [
    "Agentic AI agents use memory, tools, and goals to act.",
    "LangChain and CrewAI are popular frameworks for building AI agents.",
    "Retrieval-Augmented Generation (RAG) improves accuracy by fetching external knowledge."
]
questions = [
    "What are the key components of Agentic AI?",
    "Name one framework for AI agents.",
    "How does RAG improve answers?"
]

manual_answers = {
    questions[0]: "Agentic AI uses memory, tools, and explicit goals to act intelligently.",
    questions[1]: "LangChain is a framework for building AI agents.",
    questions[2]: "RAG improves answers by retrieving relevant external knowledge and conditioning the model on it."
}


def rag_retriever(question, context, top_k=1):
    lower_q = question.lower()
    hits = []
    for doc in context:
        score = sum(1 for token in lower_q.split() if token in doc.lower())
        hits.append((score, doc))
    hits.sort(reverse=True)
    return [doc for score, doc in hits if score > 0][:top_k] or context[:top_k]


def fine_tuning_qa(question):
    if OPENAI_API_KEY:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)

        system = "You are a fine-tuned Agentic AI expert. Base your answer solely on these facts:\n" + "\n".join(kb)
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "system", "content": system}, {"role": "user", "content": question}],
            temperature=0.1,
        )
        return response.choices[0].message.content.strip()

    return manual_answers[question] + " (fallback simulated fine-tune)"


def adapter_qa(question):
    if OPENAI_API_KEY:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)

        system = "You are an Agentic AI adapter, where the adapter prompt partially specializes a base LLM for agentic facts.\n"
        system += "The adapter profile is:\n" + "\n".join(kb)

        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "system", "content": system}, {"role": "user", "content": question}],
            temperature=0.2,
        )
        return response.choices[0].message.content.strip()

    return manual_answers[question] + " (fallback simulated adapter)"


def rag_qa(question):
    if OPENAI_API_KEY:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)

        hits = rag_retriever(question, kb, top_k=2)
        context = "Relevant KB snippets:\n" + "\n".join(f"- {h}" for h in hits)
        prompt = f"{context}\n\nQuestion: {question}\nAnswer:"
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "system", "content": "You are a domain-specialized RAG agent."}, {"role": "user", "content": prompt}],
            temperature=0.1,
        )
        return response.choices[0].message.content.strip()

    return manual_answers[question] + " (fallback RAG via keyword retrieval)"


for q in questions:
    print("Question:", q)
    print(" fine-tune:", fine_tuning_qa(q))
    print(" adapter:   ", adapter_qa(q))
    print(" RAG:       ", rag_qa(q))
    print("---")